# Assets 1 · Bronze
Loads this domain's raw CSV files into `bronze*` tables, unchanged except for load metadata. Solvi is fictional; all data is generated.

In [ ]:
# Domain: assets · owner: Service manager
# Attach this notebook to the lakehouse "Assets" in the workspace "Solvi Assets".
DOMAIN = "assets"
LANDING_PATH = f"Files/landing/{DOMAIN}"   # upload this domain's CSV files here
TABLE_FORMAT = "delta"
LOCAL_PREFIX = ""                            # local testing sets this to the domain name
AS_OF = "2026-10-31"                         # demo data ends here; use the current date with live data

from pyspark.sql import functions as F, Window as W

def write_table(df, name):
    (df.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable(LOCAL_PREFIX + name))
    print(f"wrote {name:<30} {df.count():>9,} rows")

def read_table(name):                         # a table this domain owns
    return spark.read.table(LOCAL_PREFIX + name)

def read_shared(domain, name):                # another domain's table, through a OneLake shortcut named domain + name
    return spark.read.table(f"{domain}{name}")

In [ ]:
SOURCES = ["installation", "production_daily", "service_visit", "fault_log_truth"]
load_ts = F.current_timestamp()
for src in SOURCES:
    df = (spark.read.option("header", True).option("inferSchema", False).csv(f"{LANDING_PATH}/{src}.csv")
              .withColumn("_source_file", F.lit(f"{src}.csv")).withColumn("_loaded_at", load_ts))
    write_table(df, f"bronze{src.replace('_', '')}")